# Scraper — Portal de Subastas BOE
### Paso 1: Conexión y verificación de resultados
---

## Instalación de paquetes

In [1]:
# Ejecuta esta celda solo la primera vez
# Si usas uv: abre la terminal y ejecuta:  uv add cloudscraper beautifulsoup4 lxml
# Si usas pip dentro del notebook quita el comentario de la siguiente linea:
#!pip install cloudscraper beautifulsoup4 lxml

## Imports

In [2]:
import cloudscraper
from bs4 import BeautifulSoup
import re
import time
import pandas as pd
import sys
import os
sys.path.append('../utils')
from funciones import limpiar_importe,limpiar_dataframe, filtrar_tributarios, resumen_dataset


## Crear la sesión cloudscraper

In [3]:
BASE_URL   = 'https://subastas.boe.es'
SEARCH_URL = f'{BASE_URL}/subastas_ava.php'
PAUSA_SEG = 1.5

# Crear una sesión que simula Chrome en Windows
# cloudscraper resuelve automáticamente los challenges de Cloudflare
scraper = cloudscraper.create_scraper(
    browser={'browser': 'chrome', 'platform': 'windows', 'mobile': False}
)

# Cabeceras adicionales para simular un navegador real
scraper.headers.update({
    'Accept-Language' : 'es-ES,es;q=0.9',
    'Accept'          : 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Referer'         : SEARCH_URL,
    'Origin'          : BASE_URL,
})

print('Sesión cloudscraper creada')
print(f'User-Agent: {scraper.headers["User-Agent"]}')

Sesión cloudscraper creada
User-Agent: Mozilla/5.0 (Windows NT 10.0; WOW64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.36 OPR/45.0.2552.898


## Definir los datos del formulario (POST)

***Inspeccionando cada campo que enviare en la busqueda*** <br>

Basado en la inspección del HTML, el formulario envía:

| Campo HTML | name | value | Qué hace |
|---|---|---|---|
| Radio Inmuebles | `dato[3]` | `I` | Selecciona tipo bien = Inmuebles |
| Radio subcategoría Todos | `dato[4]` | `` (vacío) | Subcategoría = Todos |
| Fecha fin desde | `dato[17][0]` | `YYYY-MM-DD` | Fecha fin subasta - inicio rango |
| Fecha fin hasta | `dato[17][1]` | `YYYY-MM-DD` | Fecha fin subasta - fin rango |
| Fecha inicio desde | `dato[18][0]` | `YYYY-MM-DD` | Fecha inicio subasta - inicio rango |
| Fecha inicio hasta | `dato[18][1]` | `YYYY-MM-DD` | Fecha inicio subasta - fin rango |
| Botón buscar | `accion` | `Buscar` | Dispara la búsqueda |

> **⚠️ Nota sobre las fechas:** el navegador envía `input type=date` en formato `YYYY-MM-DD`  
> aunque el placeholder diga `dd/mm/yyyy`. Eso es solo visual.

In [4]:
# ─── Se puede ajustar las fechas según búsqueda ────────────────────────────
# Formato YYYY-MM-DD (como lo envía el navegador internamente)

FECHA_FIN_DESDE  = '2025-10-01'   # dato[17][0]  → desdeFP  '2025-12-01'
FECHA_FIN_HASTA  = '2025-10-31'   # dato[17][1]  → hastaFP  '2025-12-01'
FECHA_INICIO_DESDE = ''           # dato[18][0]  → desdeIP  '2026-03-01'
FECHA_INICIO_HASTA = ''           # dato[18][1]  → hastaIP  '2026-04-01'

# ─── Cuerpo del POST ────────────────────────────────────────────────────────
form_data = {
    # Tipo de bien → Inmuebles
    'dato[3]'     : 'I',
    # Subcategoría → Todos (vacío = todos los inmuebles)
    'dato[4]'     : '',
    # Fecha fin subasta (rango)
    'dato[17][0]' : FECHA_FIN_DESDE,
    'dato[17][1]' : FECHA_FIN_HASTA,
    # Fecha inicio subasta (rango)
    'dato[18][0]' : FECHA_INICIO_DESDE,
    'dato[18][1]' : FECHA_INICIO_HASTA,
    # Botón de envío
    'accion'      : 'Buscar',
}

print('Formulario definido:')
for k, v in form_data.items():
    print(f'   {k:20s} = {repr(v)}')

Formulario definido:
   dato[3]              = 'I'
   dato[4]              = ''
   dato[17][0]          = '2025-10-01'
   dato[17][1]          = '2025-10-31'
   dato[18][0]          = ''
   dato[18][1]          = ''
   accion               = 'Buscar'


## Enviar el POST y verificar la respuesta

In [5]:
print('Enviando búsqueda al portal del BOE...')

response = scraper.post(SEARCH_URL, data=form_data, timeout=25)

print(f'   HTTP status  : {response.status_code}')
print(f'   URL final    : {response.url}')
print(f'   Tamaño HTML  : {len(response.text):,} caracteres')

# Verificación rápida
if response.status_code == 200:
    print('Conexión exitosa')
else:
    print(f'Error HTTP {response.status_code}')

Enviando búsqueda al portal del BOE...
   HTTP status  : 200
   URL final    : https://subastas.boe.es/subastas_ava.php
   Tamaño HTML  : 120,554 caracteres
Conexión exitosa


## Parsear HTML y verificar que llegaron resultados

In [7]:
soup = BeautifulSoup(response.text, 'lxml')

# Extraer todo el texto de la página
texto_pagina = soup.get_text(' ', strip=True)

# Buscar el texto de total: "Resultados X a Y de Z"
match = re.search(
    r'Resultados\s+(\d+)\s+a\s+(\d+)\s+de\s+([\d\.]+)',
    texto_pagina,
    re.IGNORECASE
)

if match:
    desde = int(match.group(1))
    hasta = int(match.group(2))
    total_str = match.group(3).replace('.', '')
    total = int(total_str)
    num_paginas = (total // 50) + 1

    print('==================================')
    print(f'Resultados {desde} a {hasta} de {total:,}')
    print(f'Páginas estimadas (50/pág): {num_paginas}')
    print('==================================')
else:
    print('No se encontró el texto de resultados.')
    print('\n--- Diagnóstico: primeros 2000 caracteres del HTML ---')
    print(texto_pagina[:2000])

Resultados 1 a 50 de 1,671
Páginas estimadas (50/pág): 34


In [8]:
def extraer_id_busqueda(soup):
    """
    Extrae el token base del id_busqueda desde el link de paginación.
    Ejemplo href: subastas_ava.php?accion=Mas&id_busqueda=<TOKEN>,-50-50
    Nos quedamos solo con <TOKEN> (sin el sufijo ,-N-50).
    """
    link_pag2 = soup.find('a', string='2')
    if link_pag2:
        href = link_pag2.get('href', '')
        m = re.search(r'id_busqueda=(.+?),-\d+-\d+', href)
        if m:
            return m.group(1)
    return None

id_busqueda = extraer_id_busqueda(soup)
if id_busqueda:
    print(f'Token extraído: {id_busqueda[:40]}...')
else:
    print('No encontrado')

Token extraído: cjhmcG5jVFZDQlVuRzdqblBYNlNBa2laWUVITFRW...


In [9]:
def extraer_listado(soup):
    """
    Extrae datos de cada <li class='resultado-busqueda'>.

    Estructura HTML:
      <li class='resultado-busqueda'>
        <h3>SUBASTA SUB-XX-...</h3>          ← referencia
        <h4>ORGANISMO - CIUDAD</h4>          ← organismo
        <p>Expediente: ...</p>               ← opcional
        <p>Estado: ... [Fecha...]</p>
        <p>Descripción del bien</p>
        <a class='resultado-busqueda-link-otro' href='...'>Más...</a>
      </li>
    """
    registros = []
    items = soup.find_all('li', class_='resultado-busqueda')

    for item in items:
        reg = {}

        # Referencia (h3) — puede venir con "(2 lotes)"
        h3 = item.find('h3')
        if h3:
            texto_h3 = h3.get_text(strip=True)
            m_ref = re.search(r'(SUB-[A-Z0-9\-]+)', texto_h3)
            reg['referencia'] = m_ref.group(1) if m_ref else texto_h3
            m_lotes = re.search(r'\((\d+) lotes?\)', texto_h3, re.I)
            reg['num_lotes_listado'] = int(m_lotes.group(1)) if m_lotes else 1

        # Organismo (h4)
        h4 = item.find('h4')
        reg['organismo'] = h4.get_text(strip=True) if h4 else None

        # Párrafos: expediente, estado, descripción
        reg['expediente']             = None
        reg['estado']                 = None
        reg['fecha_conclusion_listado'] = None
        reg['descripcion']            = None

        for p in item.find_all('p'):
            texto_p = p.get_text(' ', strip=True)
            if not texto_p:
                continue
            if texto_p.startswith('Expediente:'):
                reg['expediente'] = texto_p.replace('Expediente:', '').strip()
            elif texto_p.startswith('Estado:'):
                m_estado = re.match(r'Estado:\s*([^\[\-]+)', texto_p)
                reg['estado'] = m_estado.group(1).strip() if m_estado else texto_p
                m_fecha = re.search(r'Fecha de conclusión:\s*([\d/]+ a las [\d:]+)', texto_p)
                reg['fecha_conclusion_listado'] = m_fecha.group(1) if m_fecha else None
            else:
                reg['descripcion'] = texto_p  # último párrafo no vacío

        # URL del detalle + id_sub limpio
        enlace = item.find('a', class_='resultado-busqueda-link-otro')
        if enlace:
            href = enlace['href'].replace('./', '')
            reg['url_detalle'] = f"{BASE_URL}/{href}"
            m_idsub = re.search(r'idSub=([^&]+)', href)
            reg['id_sub'] = m_idsub.group(1) if m_idsub else None

        registros.append(reg)

    return registros


# Prueba con la primera página
registros_p1 = extraer_listado(soup)
print(f'Tarjetas extraídas en página 1: {len(registros_p1)}')
print('\nEjemplo — primer registro:')
for k, v in registros_p1[0].items():
    print(f'  {k:30s}: {v}')

Tarjetas extraídas en página 1: 50

Ejemplo — primer registro:
  referencia                    : SUB-JA-2024-231148
  num_lotes_listado             : 1
  organismo                     : JUZGADO 1 INSTANCIA 2 - LEON
  expediente                    : 0146/21
  estado                        : Pendiente de finalización y devolución de depósitos con reserva
  fecha_conclusion_listado      : 01/10/2025 a las 18:00:00
  descripcion                   : FINCA NÚMERO DIEZ.- Vivienda en la planta cuarta de la
          casa sita en León, a la calle de Los Templarios, número cinco,
          que es la de la izquierda subiendo la escalera. Tiene una
          superficie útil, incluyendo la carbonera que le es aneja, de
          SETENTA Y OCHO METROS Y SESENTA Y TRES DECÍMETROS CUADRADOS.
          Linda, tomando como frente la calle de su situación: derecha
          entrando, finca de don Aquilino Bodelón; izquierda, vivienda
          derecha de la misma planta, rellano, caja de escalera y patio

In [10]:
def extraer_detalle(scraper, url):
    """
    Descarga la ficha individual y extrae la tabla <th>/<td>.

    Campos que extrae:
      tipo_subasta, cuenta_expediente, fecha_inicio, fecha_conclusion,
      cantidad_reclamada_eur, lotes, anuncio_boe,
      valor_subasta_eur, tasacion_eur, puja_minima_eur,
      tramo_pujas_eur, deposito_eur
    """
    try:
        r = scraper.get(url, timeout=20)
        r.raise_for_status()
    except Exception as e:
        print(f'  ⚠️  Error: {e}')
        return {}

    soup_det = BeautifulSoup(r.text, 'lxml')
    datos = {}

    MAPA_CAMPOS = {
        'Identificador'       : 'referencia_det',
        'Tipo de subasta'     : 'tipo_subasta',
        'Cuenta expediente'   : 'cuenta_expediente',
        'Fecha de inicio'     : 'fecha_inicio',
        'Fecha de conclusión' : 'fecha_conclusion',
        'Cantidad reclamada'  : 'cantidad_reclamada_eur',
        'Lotes'               : 'lotes',
        'Forma adjudicación'  : 'forma_adjudicacion',
        'Anuncio BOE'         : 'anuncio_boe',
        'Valor subasta'       : 'valor_subasta_eur',
        'Tasación'            : 'tasacion_eur',
        'Puja mínima'         : 'puja_minima_eur',
        'Tramos entre pujas'  : 'tramo_pujas_eur',
        'Importe del depósito': 'deposito_eur',
    }

    for fila in soup_det.find_all('tr'):
        th = fila.find('th')
        td = fila.find('td')
        if not th or not td:
            continue

        clave = th.get_text(strip=True)
        valor = td.get_text(' ', strip=True)
        col   = MAPA_CAMPOS.get(clave)

        if col is None:
            continue

        if col.endswith('_eur'):
            datos[col] = limpiar_importe(valor)
        elif col.startswith('fecha_'):
            # '11-11-2025 18:00:00 CET (ISO: ...)' → '11-11-2025 18:00:00 CET'
            datos[col] = valor.split('(')[0].strip()
        elif col == 'lotes':
            #Si lotes contienve el texto 'sin' datos[lotes] = 1
            datos[col] = 1 if 'sin' in valor.lower() else int(re.search(r'\d+', valor).group())
        else:
            datos[col] = valor

    return datos


# Prueba con la 4ª subasta (vivienda de Madrid que viste en el inspector)
url_prueba = registros_p1[3]['url_detalle']
print(f'Probando detalle...')
detalle_prueba = extraer_detalle(scraper, url_prueba)
print('Datos del detalle:')
for k, v in detalle_prueba.items():
    print(f'  {k:25s}: {v}')

Probando detalle...
Datos del detalle:
  referencia_det           : SUB-JA-2025-244592
  tipo_subasta             : JUDICIAL EN VÍA DE APREMIO
  cuenta_expediente        : 4182 0000 06 0442 20
  cantidad_reclamada_eur   : 205737.8
  lotes                    : 2
  forma_adjudicacion       : Separada para cada lote
  anuncio_boe              : BOE-B-2025-31954
  valor_subasta_eur        : None
  tasacion_eur             : None
  puja_minima_eur          : None
  tramo_pujas_eur          : None
  deposito_eur             : None


In [13]:
def get_pagina_listado(scraper, n_pagina, id_busqueda):
    """
    Paginación real del BOE:
      página 1 → ,-0-50
      página 2 → ,-50-50
      página 3 → ,-100-50
      página N → ,-(N-1)*50-50
    """
    offset = (n_pagina - 1) * 50
    id_busqueda_completo = f'{id_busqueda},-{offset}-50'

    url = f'{SEARCH_URL}?accion=Mas&id_busqueda={id_busqueda_completo}'

    try:
        r = scraper.get(url, timeout=20)
        r.raise_for_status()
        return BeautifulSoup(r.text, 'lxml')
    except Exception as e:
        print(f'Error en página {n_pagina}: {e}')
        return None


# Prueba: página 2
soup_p2 = get_pagina_listado(scraper, n_pagina=2, id_busqueda=id_busqueda)
if soup_p2:
    regs_p2 = extraer_listado(soup_p2)
    print(f'Página 2: {len(regs_p2)} tarjetas')
    print(f'Primera : {regs_p2[0]["referencia"]}')
    print(f'Última  : {regs_p2[-1]["referencia"]}')

print()

Página 2: 50 tarjetas
Primera : SUB-JA-2025-244182
Última  : SUB-AT-2025-25R0686001040



In [14]:
MAX_PAGINAS = num_paginas  # ← Inicialmentesubir a num_paginas cuando valides que funciona

todos_registros = []

for n_pag in range(1, MAX_PAGINAS + 1):
    print(f'\nPágina {n_pag}/{MAX_PAGINAS}')

    if n_pag == 1:
        soup_pag = soup  # ya descargada
    else:
        posicion = (n_pag - 1) * 50 + 1
        soup_pag = get_pagina_listado(scraper, n_pagina=n_pag, id_busqueda=id_busqueda)
        if soup_pag is None:
            print(f'Sin respuesta, saltando')
            continue
        time.sleep(PAUSA_SEG)

    registros_listado = extraer_listado(soup_pag)
    print(f'  Tarjetas: {len(registros_listado)}')

    for i, reg in enumerate(registros_listado):
        url = reg.get('url_detalle')
        if url:
            detalle = extraer_detalle(scraper, url)
            reg.update(detalle)
            time.sleep(PAUSA_SEG)
        todos_registros.append(reg)
        if (i + 1) % 10 == 0:
            print(f'    → {i+1}/{len(registros_listado)} procesados')

print(f'\n === Total registros: {len(todos_registros)} ===')


Página 1/34
  Tarjetas: 50
    → 10/50 procesados
    → 20/50 procesados
    → 30/50 procesados
    → 40/50 procesados
    → 50/50 procesados

Página 2/34
  Tarjetas: 50
    → 10/50 procesados
    → 20/50 procesados
    → 30/50 procesados
    → 40/50 procesados
    → 50/50 procesados

Página 3/34
  Tarjetas: 50
    → 10/50 procesados
    → 20/50 procesados
    → 30/50 procesados
    → 40/50 procesados
    → 50/50 procesados

Página 4/34
  Tarjetas: 50
    → 10/50 procesados
    → 20/50 procesados
    → 30/50 procesados
    → 40/50 procesados
    → 50/50 procesados

Página 5/34
  Tarjetas: 50
    → 10/50 procesados
    → 20/50 procesados
    → 30/50 procesados
    → 40/50 procesados
    → 50/50 procesados

Página 6/34
  Tarjetas: 50
    → 10/50 procesados
    → 20/50 procesados
    → 30/50 procesados
    → 40/50 procesados
    → 50/50 procesados

Página 7/34
  Tarjetas: 50
    → 10/50 procesados
    → 20/50 procesados
    → 30/50 procesados
    → 40/50 procesados
    → 50/50 procesados

In [15]:
df = pd.DataFrame(todos_registros).copy()
df

,referencia,num_lotes_listado,organismo,expediente,estado,fecha_conclusion_listado,descripcion,url_detalle,id_sub,referencia_det,...,fecha_conclusion,cantidad_reclamada_eur,lotes,anuncio_boe,valor_subasta_eur,tasacion_eur,puja_minima_eur,tramo_pujas_eur,deposito_eur,forma_adjudicacion
0,SUB-JA-2024-231148,1,JUZGADO 1 INSTANCIA 2 - LEON,0146/21,Pendiente de finalización y devolución de depó...,01/10/2025 a las 18:00:00,FINCA NÚMERO DIEZ.- Vivienda en la planta cuar...,https://subastas.boe.es/detalleSubasta.php?idS...,SUB-JA-2024-231148,SUB-JA-2024-231148,...,01-10-2025 18:00:00 CET,38074.96,1,BOE-B-2025-31964,110887.98,0.00,NaN,NaN,5544.40,NaN
1,SUB-JA-2025-237883,1,UNIDAD SUBASTAS JUDICIALES MURCIA - MURCIA,0353/17,Finalizada y depósitos con reserva devueltos,01/10/2025 a las 18:00:00,Id. lote. 237883L1. Finca registral 37881. Pis...,https://subastas.boe.es/detalleSubasta.php?idS...,SUB-JA-2025-237883,SUB-JA-2025-237883,...,01-10-2025 18:00:00 CET,56189.91,1,BOE-B-2025-31986,107904.68,0.00,NaN,2158.10,5395.23,NaN
2,SUB-JA-2025-242137,4,JUZGADO 1 INST E INSTRUCC. 1 - SOLSONA,0006/20,Finalizada y depósitos con reserva devueltos,01/10/2025 a las 18:00:00,"Subasta con varios lotes. Lote 1: URBANA, Parc...",https://subastas.boe.es/detalleSubasta.php?idS...,SUB-JA-2025-242137,SUB-JA-2025-242137,...,01-10-2025 18:00:00 CET,112641.00,4,BOE-B-2025-31977,NaN,NaN,NaN,NaN,NaN,Separada para cada lote
3,SUB-JA-2025-244592,2,JUZGADO 1 INST E INSTRUCC. 1 - GANDESA,0442/20,Cancelada,NaN,Subasta con varios lotes. Lote 1: 1. URBANA. C...,https://subastas.boe.es/detalleSubasta.php?idS...,SUB-JA-2025-244592,SUB-JA-2025-244592,...,NaN,205737.80,2,BOE-B-2025-31954,NaN,NaN,NaN,NaN,NaN,Separada para cada lote
4,SUB-JA-2025-245672,1,Sección Civil e Instrucción TI Blanes. Plz.n 1...,0719/20,Finalizada y depósitos con reserva devueltos,01/10/2025 a las 18:00:00,URBANA: FINCA ESPECIAL. ENTIDAD NUMERO TRES.- ...,https://subastas.boe.es/detalleSubasta.php?idS...,SUB-JA-2025-245672,SUB-JA-2025-245672,...,01-10-2025 18:00:00 CET,179629.82,1,BOE-B-2025-31945,259600.00,0.00,NaN,5192.00,12980.00,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1666,SUB-JA-2025-252484,1,JUZGADO 1 INSTANCIA 9 - CORDOBA,0307/24,Pendiente de finalización y devolución de depó...,30/10/2025 a las 23:18:38,URBANA. - NÚMERO DOCE. - Apartamento bajo tipo...,https://subastas.boe.es/detalleSubasta.php?idS...,SUB-JA-2025-252484,SUB-JA-2025-252484,...,30-10-2025 23:18:38 CET,86355.15,1,BOE-B-2025-36086,110000.00,0.00,NaN,2200.00,5500.00,NaN
1667,SUB-JV-2025-250646,2,SERVICIO COMUN GENERAL SUBASTAS - VALENCIA,0218/21,Finalizada y depósitos con reserva devueltos,31/10/2025 a las 02:03:50,Subasta con varios lotes. Lote 1: Vivienda en ...,https://subastas.boe.es/detalleSubasta.php?idS...,SUB-JV-2025-250646,SUB-JV-2025-250646,...,31-10-2025 02:03:50 CET,594526.61,2,BOE-B-2025-36109,NaN,NaN,NaN,NaN,NaN,Separada para cada lote
1668,SUB-JA-2025-247052,1,JUZGADO 1 INSTANCIA 73 - MADRID,0065/17,Finalizada y depósitos con reserva devueltos,31/10/2025 a las 03:04:41,"VIVIIENDA URBANA PISO 4º, PLANTA 1ª, DEL Nº 20...",https://subastas.boe.es/detalleSubasta.php?idS...,SUB-JA-2025-247052,SUB-JA-2025-247052,...,31-10-2025 03:04:41 CET,11198.68,1,BOE-B-2025-36097,75112.98,305561.57,NaN,1502.25,3755.64,NaN
1669,SUB-JC-2025-251590,1,SERVICIO COMUN GENERAL SUBASTAS - VALENCIA,0745/21,Pendiente de finalización y devolución de depó...,31/10/2025 a las 08:43:54,"VIVIENDA SITA EN VALENCIA CALLE JURATS, 29-20ª",https://subastas.boe.es/detalleSubasta.php?idS...,SUB-JC-2025-251590,SUB-JC-2025-251590,...,31-10-2025 08:43:54 CET,79714.00,1,BOE-B-2025-36110,79714.00,0.00,NaN,NaN,3985.70,NaN


In [16]:
df = pd.DataFrame(todos_registros).copy()

# Convertir fechas
for col in ['fecha_inicio', 'fecha_conclusion']:
    if col in df.columns:
        df[col] = pd.to_datetime(
            df[col].str.extract(r'(\d{2}-\d{2}-\d{4})')[0],
            format='%d-%m-%Y',
            errors='coerce'
        )
   
print(f'Shape: {df.shape}')
print(f'Columnas: {list(df.columns)}')
df.head(3)

Shape: (1671, 23)
Columnas: ['referencia', 'num_lotes_listado', 'organismo', 'expediente', 'estado', 'fecha_conclusion_listado', 'descripcion', 'url_detalle', 'id_sub', 'referencia_det', 'tipo_subasta', 'cuenta_expediente', 'fecha_inicio', 'fecha_conclusion', 'cantidad_reclamada_eur', 'lotes', 'anuncio_boe', 'valor_subasta_eur', 'tasacion_eur', 'puja_minima_eur', 'tramo_pujas_eur', 'deposito_eur', 'forma_adjudicacion']


,referencia,num_lotes_listado,organismo,expediente,estado,fecha_conclusion_listado,descripcion,url_detalle,id_sub,referencia_det,...,fecha_conclusion,cantidad_reclamada_eur,lotes,anuncio_boe,valor_subasta_eur,tasacion_eur,puja_minima_eur,tramo_pujas_eur,deposito_eur,forma_adjudicacion
0,SUB-JA-2024-231148,1,JUZGADO 1 INSTANCIA 2 - LEON,0146/21,Pendiente de finalización y devolución de depó...,01/10/2025 a las 18:00:00,FINCA NÚMERO DIEZ.- Vivienda en la planta cuar...,https://subastas.boe.es/detalleSubasta.php?idS...,SUB-JA-2024-231148,SUB-JA-2024-231148,...,2025-10-01,38074.96,1,BOE-B-2025-31964,110887.98,0.0,NaN,NaN,5544.40,NaN
1,SUB-JA-2025-237883,1,UNIDAD SUBASTAS JUDICIALES MURCIA - MURCIA,0353/17,Finalizada y depósitos con reserva devueltos,01/10/2025 a las 18:00:00,Id. lote. 237883L1. Finca registral 37881. Pis...,https://subastas.boe.es/detalleSubasta.php?idS...,SUB-JA-2025-237883,SUB-JA-2025-237883,...,2025-10-01,56189.91,1,BOE-B-2025-31986,107904.68,0.0,NaN,2158.1,5395.23,NaN
2,SUB-JA-2025-242137,4,JUZGADO 1 INST E INSTRUCC. 1 - SOLSONA,0006/20,Finalizada y depósitos con reserva devueltos,01/10/2025 a las 18:00:00,"Subasta con varios lotes. Lote 1: URBANA, Parc...",https://subastas.boe.es/detalleSubasta.php?idS...,SUB-JA-2025-242137,SUB-JA-2025-242137,...,2025-10-01,112641.00,4,BOE-B-2025-31977,NaN,NaN,NaN,NaN,NaN,Separada para cada lote


In [17]:
df.to_csv('../data/subastas_inmuebles_oct2025.csv', index=False, encoding='utf-8-sig') #subastas_inmuebles_dic2025.csv
print(f'Guardado: {len(df)} filas, {df.shape[1]} columnas')
print()
print(df.dtypes)

Guardado: 1671 filas, 23 columnas

referencia                             str
num_lotes_listado                    int64
organismo                              str
expediente                             str
estado                                 str
fecha_conclusion_listado               str
descripcion                            str
url_detalle                            str
id_sub                                 str
referencia_det                         str
tipo_subasta                           str
cuenta_expediente                      str
fecha_inicio                datetime64[us]
fecha_conclusion            datetime64[us]
cantidad_reclamada_eur             float64
lotes                                int64
anuncio_boe                            str
valor_subasta_eur                  float64
tasacion_eur                       float64
puja_minima_eur                    float64
tramo_pujas_eur                    float64
deposito_eur                       float64
forma_adjudicacion 